# Manga Translator — free GPU backend (Colab)

Runs the **backend** pipeline on a free Colab **T4 GPU** and exposes it on a public URL via a Cloudflare quick tunnel. On GPU, detection + LaMa inpainting drop from ~25s to ~1-2s, so a panel goes from ~30s to ~8s (now floored by the Claude API call).

**Before running:** `Runtime -> Change runtime type -> T4 GPU`.

The same `backend/` code runs here unchanged — it auto-detects CUDA (`detect.py` `use_gpu`, `inpaint.py` device).

In [ ]:
# 1. Get the code: upload backend_for_colab.zip when prompted.
#    (Alternative: git-clone your repo into /content/manga instead.)
from google.colab import files
import zipfile, os
up = files.upload()  # pick backend_for_colab.zip from your machine
with zipfile.ZipFile(next(iter(up))) as z:
    z.extractall("/content/manga")
BACKEND = "/content/manga/backend"
print("backend:", BACKEND, "->", os.listdir(BACKEND))

In [ ]:
# 2. Install deps. Colab ships CUDA torch already (LaMa uses it automatically).
#    paddlepaddle-gpu lights up GPU detection; if it fails, detection falls
#    back to CPU and everything still works.
!pip -q install "paddleocr<3" simple-lama-inpainting fastapi uvicorn anthropic \
    python-dotenv python-multipart httpx opencv-python nest_asyncio
!pip -q install paddlepaddle-gpu==2.6.2 || echo "paddle-gpu install failed -> detection on CPU"
import torch; print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
# 3. Anthropic API key (typed, not stored in the notebook).
import getpass
key = getpass.getpass("ANTHROPIC_API_KEY: ")
with open(os.path.join(BACKEND, ".env"), "w") as f:
    f.write(f"ANTHROPIC_API_KEY={key}\n")
print("wrote backend/.env")

In [ ]:
# 4. Start the backend (background thread) + a public Cloudflare tunnel.
import os, re, time, threading, subprocess
os.chdir(BACKEND)

import nest_asyncio, uvicorn
nest_asyncio.apply()
threading.Thread(
    target=lambda: uvicorn.run("main:app", host="0.0.0.0", port=8000, log_level="info"),
    daemon=True,
).start()

# cloudflared quick tunnel — no signup needed.
if not os.path.exists("/content/cloudflared"):
    subprocess.run(["wget", "-q",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "-O", "/content/cloudflared"], check=True)
    subprocess.run(["chmod", "+x", "/content/cloudflared"], check=True)

proc = subprocess.Popen(
    ["/content/cloudflared", "tunnel", "--url", "http://localhost:8000", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
public_url = None
for line in proc.stdout:
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", line)
    if m:
        public_url = m.group(0)
        break
print("\nPUBLIC URL:", public_url)
print("Health check:", public_url + "/health")

In [ ]:
# 5. Smoke test: translate one image through the GPU backend.
#    Upload a panel to /content first, or use the test helper.
import urllib.request, json, base64, uuid, time
SRC = "/content/panel.jpg"  # <-- upload an image here

img = open(SRC, "rb").read()
b = uuid.uuid4().hex
body = (f'--{b}\r\nContent-Disposition: form-data; name="file"; filename="x.jpg"\r\n'
        f'Content-Type: image/jpeg\r\n\r\n').encode() + img + f"\r\n--{b}--\r\n".encode()
req = urllib.request.Request(public_url + "/translate-image", data=body,
    headers={"Content-Type": f"multipart/form-data; boundary={b}"})
t = time.time()
d = json.loads(urllib.request.urlopen(req, timeout=300).read())
print(f"done in {time.time()-t:.1f}s | {len(d['regions'])} bubbles")
for r in d["regions"]:
    print("  -", (r["translated_text"] or "")[:60])
open("/content/out.png", "wb").write(base64.b64decode(d["rendered_image"].split(",",1)[1]))
print("wrote /content/out.png")

## Point the frontend at the GPU backend

Set the Vite dev proxy target (in `frontend/vite.config.js`) to the printed `PUBLIC URL`, or call it directly from the browser. The tunnel URL changes each run.

**Limits:** Colab free sessions are ephemeral (~90 min idle, ~12h max) — this is a dev/demo accelerator, not 24/7 hosting. For more hours, Kaggle (30h/week) runs the same notebook.